# 73 Cross-Run City Swap Comparison

One notebook to compare all city-swap runs we already have:
- `70_city_swap_batch_eval.ipynb` main models
- `72_restored_models_city_swap.ipynb` restored adversarial / attr_reg attempts
- `challengers/c10_challenger_city_swap_eval.ipynb` selected challenger winners

Goal: compare existing swap evidence without retraining anything.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ModuleNotFoundError:
    HAS_MPL = False

CWD = Path.cwd()
NOTEBOOK_DIR = CWD if (CWD / "results").exists() else (CWD / "notebooks")
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR.parent

MAIN_SWAP_JSON = REPO_ROOT / "notebooks" / "results" / "city_swap_all" / "city_swap_all_models.json"
UNIFIED_TABLE_CSV = REPO_ROOT / "notebooks" / "results" / "unified_comparison" / "c71_unified_models_table.csv"
C10_SUMMARY_CSV = REPO_ROOT / "results" / "challengers_city_swap" / "c10_selected_family_city_swap" / "c10_selected_family_city_swap_summary.csv"
R72_SUMMARY_CSV = REPO_ROOT / "notebooks" / "results" / "two-models-restore" / "city_swap_restored_models" / "72_restored_models_city_swap_summary.csv"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "results" / "cross_run_city_swap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Output dir:", OUTPUT_DIR)


In [ ]:
unified = pd.read_csv(UNIFIED_TABLE_CSV)
unified = unified.rename(columns={"display_name": "display_name_c71"})

with MAIN_SWAP_JSON.open("r", encoding="utf-8") as f:
    main_swap_raw = json.load(f)

main_rows = []
for run_name, payload in main_swap_raw.items():
    row = {
        "source_run": "70_batch",
        "expected_track": "main",
        "run_name": run_name,
        "status": "error" if "error" in payload else "ok",
        "error": payload.get("error"),
        "model_dir": payload.get("model_dir"),
        "overall_flip_rate": payload.get("overall_flip_rate"),
        "swap_accuracy": payload.get("accuracy"),
        "swap_macro_f1": payload.get("macro_f1"),
        "uses_scrubbing": payload.get("uses_scrubbing"),
    }
    if row["model_dir"]:
        row["model_name"] = Path(row["model_dir"]).name
    else:
        row["model_name"] = run_name

    per_city = payload.get("per_city_flip_rate", {})
    per_class = payload.get("per_class_flip_rate", {})
    if per_city:
        city_rates = [v.get("flip_rate", v) if isinstance(v, dict) else v for v in per_city.values()]
        row["max_city_flip_rate"] = max(city_rates)
        row["min_city_flip_rate"] = min(city_rates)
    if per_class:
        class_rates = [v.get("flip_rate", v) if isinstance(v, dict) else v for v in per_class.values()]
        row["max_class_flip_rate"] = max(class_rates)
    main_rows.append(row)

main_df = pd.DataFrame(main_rows)

c10_df = pd.read_csv(C10_SUMMARY_CSV).rename(
    columns={
        "family": "family_from_swap",
        "accuracy": "swap_accuracy",
        "macro_f1": "swap_macro_f1",
    }
)
c10_df["source_run"] = "c10_challengers"
c10_df["expected_track"] = "challenger"
c10_df["run_name"] = c10_df["model_name"]
c10_df["status"] = c10_df["status"].fillna("ok")
city_flip_cols = [c for c in c10_df.columns if c.startswith("flip_")]
if city_flip_cols:
    c10_df["max_city_flip_rate"] = c10_df[city_flip_cols].max(axis=1)
    c10_df["min_city_flip_rate"] = c10_df[city_flip_cols].min(axis=1)

restored_df = pd.read_csv(R72_SUMMARY_CSV).rename(
    columns={
        "family": "family_from_swap",
        "accuracy": "swap_accuracy",
        "macro_f1": "swap_macro_f1",
    }
)
restored_df["source_run"] = "72_restored"
restored_df["expected_track"] = "main"
restored_df["run_name"] = restored_df["model_name"]

keep_cols = [
    "source_run",
    "run_name",
    "expected_track",
    "model_name",
    "family_from_swap",
    "status",
    "error",
    "swap_accuracy",
    "swap_macro_f1",
    "overall_flip_rate",
    "max_city_flip_rate",
    "min_city_flip_rate",
    "model_dir",
    "model_path",
    "uses_scrubbing",
]

main_df = main_df.reindex(columns=keep_cols)
c10_df = c10_df.reindex(columns=keep_cols)
restored_df = restored_df.reindex(columns=keep_cols)

swap_all = pd.concat([main_df, c10_df, restored_df], ignore_index=True, sort=False)
swap_all


In [ ]:
compare_df = swap_all.merge(
    unified[
        [
            "track",
            "family_group",
            "model_name",
            "display_name_c71",
            "method",
            "accuracy",
            "macro_f1",
            "worst_gap",
            "macro_gap",
        ]
    ],
    left_on=["model_name", "expected_track"],
    right_on=["model_name", "track"],
    how="left",
)

compare_df["family"] = compare_df["family_from_swap"].fillna(compare_df["family_group"])
compare_df["display_name"] = compare_df["display_name_c71"].fillna(compare_df["model_name"])
compare_df["rank_flip_then_f1"] = compare_df.sort_values(
    ["overall_flip_rate", "swap_macro_f1"],
    ascending=[True, False],
    na_position="last",
).reset_index().index + 1
compare_df["rank_gap_then_f1"] = compare_df.sort_values(
    ["worst_gap", "macro_f1"],
    ascending=[True, False],
    na_position="last",
).reset_index().index + 1

compare_df = compare_df[
    [
        "rank_flip_then_f1",
        "rank_gap_then_f1",
        "source_run",
        "track",
        "family",
        "model_name",
        "display_name",
        "status",
        "accuracy",
        "macro_f1",
        "worst_gap",
        "macro_gap",
        "swap_accuracy",
        "swap_macro_f1",
        "overall_flip_rate",
        "max_city_flip_rate",
        "min_city_flip_rate",
        "uses_scrubbing",
        "error",
    ]
].sort_values(["status", "overall_flip_rate", "swap_macro_f1"], ascending=[True, True, False], na_position="last")

compare_df.to_csv(OUTPUT_DIR / "73_cross_run_city_swap_table.csv", index=False)
compare_df


In [ ]:
ok_df = compare_df[compare_df["status"] == "ok"].copy()

summary = {
    "n_total_rows": int(len(compare_df)),
    "n_ok_rows": int((compare_df["status"] == "ok").sum()),
    "n_failed_rows": int((compare_df["status"] != "ok").sum()),
}

if not ok_df.empty:
    best_flip = ok_df.sort_values(["overall_flip_rate", "swap_macro_f1"], ascending=[True, False]).iloc[0]
    best_macro_f1 = ok_df.sort_values(["swap_macro_f1", "overall_flip_rate"], ascending=[False, True]).iloc[0]
    summary["best_city_swap_stability"] = {
        "model_name": best_flip["model_name"],
        "source_run": best_flip["source_run"],
        "overall_flip_rate": None if pd.isna(best_flip["overall_flip_rate"]) else float(best_flip["overall_flip_rate"]),
        "swap_macro_f1": None if pd.isna(best_flip["swap_macro_f1"]) else float(best_flip["swap_macro_f1"]),
    }
    summary["best_swap_macro_f1"] = {
        "model_name": best_macro_f1["model_name"],
        "source_run": best_macro_f1["source_run"],
        "overall_flip_rate": None if pd.isna(best_macro_f1["overall_flip_rate"]) else float(best_macro_f1["overall_flip_rate"]),
        "swap_macro_f1": None if pd.isna(best_macro_f1["swap_macro_f1"]) else float(best_macro_f1["swap_macro_f1"]),
    }

by_source = (
    compare_df.groupby("source_run", dropna=False)
    .agg(
        rows=("model_name", "size"),
        ok_rows=("status", lambda s: int((s == "ok").sum())),
        median_flip=("overall_flip_rate", "median"),
        best_flip=("overall_flip_rate", "min"),
        best_swap_f1=("swap_macro_f1", "max"),
    )
    .reset_index()
)
by_source.to_csv(OUTPUT_DIR / "73_cross_run_city_swap_by_source.csv", index=False)

report_lines = [
    "=== 73 cross-run city-swap comparison ===",
    f"rows_total: {summary['n_total_rows']}",
    f"rows_ok: {summary['n_ok_rows']}",
    f"rows_failed: {summary['n_failed_rows']}",
    "",
]
if "best_city_swap_stability" in summary:
    item = summary["best_city_swap_stability"]
    report_lines.extend(
        [
            "best_city_swap_stability:",
            f"  model_name: {item['model_name']}",
            f"  source_run: {item['source_run']}",
            f"  overall_flip_rate: {item['overall_flip_rate']}",
            f"  swap_macro_f1: {item['swap_macro_f1']}",
            "",
        ]
    )
if "best_swap_macro_f1" in summary:
    item = summary["best_swap_macro_f1"]
    report_lines.extend(
        [
            "best_swap_macro_f1:",
            f"  model_name: {item['model_name']}",
            f"  source_run: {item['source_run']}",
            f"  overall_flip_rate: {item['overall_flip_rate']}",
            f"  swap_macro_f1: {item['swap_macro_f1']}",
            "",
        ]
    )

failed = compare_df[compare_df["status"] != "ok"][["model_name", "source_run", "status", "error"]]
if not failed.empty:
    report_lines.append("failed_rows:")
    for _, row in failed.iterrows():
        report_lines.append(
            f"  - {row['model_name']} | source={row['source_run']} | status={row['status']} | error={row['error']}"
        )

report_text = "\n".join(report_lines)
print(report_text)
(OUTPUT_DIR / "73_cross_run_city_swap_report.txt").write_text(report_text, encoding="utf-8")
(OUTPUT_DIR / "73_cross_run_city_swap_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
by_source
        


In [ ]:
leaderboard_cols = [
    "source_run",
    "family",
    "model_name",
    "status",
    "accuracy",
    "macro_f1",
    "swap_accuracy",
    "swap_macro_f1",
    "overall_flip_rate",
    "worst_gap",
    "macro_gap",
]
leaderboard = compare_df[leaderboard_cols].copy()
leaderboard


In [ ]:
if HAS_MPL:
    plot_df = compare_df[compare_df["status"] == "ok"].copy()
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = {"70_batch": "#1f77b4", "c10_challengers": "#2ca02c", "72_restored": "#d62728"}
    for source_run, chunk in plot_df.groupby("source_run"):
        ax.scatter(
            chunk["overall_flip_rate"],
            chunk["swap_macro_f1"],
            s=70,
            label=source_run,
            color=colors.get(source_run, "#7f7f7f"),
            alpha=0.85,
        )
        for _, row in chunk.iterrows():
            ax.annotate(row["model_name"], (row["overall_flip_rate"], row["swap_macro_f1"]), fontsize=8, alpha=0.8)

    ax.set_xlabel("Overall city-swap flip rate")
    ax.set_ylabel("Macro-F1 on swapped set")
    ax.set_title("Cross-run city-swap comparison")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "73_cross_run_city_swap_scatter.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("matplotlib is not installed; skipping scatter plot.")
